# Chỉ khớp môi (LatentSync 1.5) — giọng đã có sẵn

1. Cột phải → **Session options** → Accelerator **GPU T4 x1**, Internet **On**.
2. **Add Input → Upload → New Dataset**: tải lên `giong.wav` và `mat.mp4`.
3. **Run All** (~10 phút cài + ~20–30 phút chạy).
4. Tải **`noi.mp4`** ở mục Output → bỏ vào Google Drive → gửi link cho Claude.

In [ ]:
import glob, shutil, os
def find(n):
    h = glob.glob(f"/kaggle/input/**/{n}", recursive=True); assert h, f"Không thấy {n}"; return h[0]
os.makedirs("/kaggle/working", exist_ok=True)
shutil.copy(find("giong.wav"), "/kaggle/working/giong.wav")
print(find("mat.mp4"))
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%bash
set -e
mkdir -p /kaggle/temp && cd /kaggle/temp
pip install -q uv
export UV_LINK_MODE=copy
[ -d LatentSync ] || git clone -q --depth 1 https://github.com/bytedance/LatentSync
[ -d lsenv ] || uv venv -q -p 3.10 lsenv
uv pip install -q -p lsenv/bin/python -r LatentSync/requirements.txt --index-strategy unsafe-best-match
cd LatentSync
../lsenv/bin/huggingface-cli download ByteDance/LatentSync-1.5 whisper/tiny.pt latentsync_unet.pt --local-dir checkpoints >/dev/null
ls -la checkpoints
echo "CAI DAT XONG"

In [ ]:
%%bash
set -e
cd /kaggle/temp/LatentSync
ffmpeg -v error -y -i /kaggle/working/giong.wav -ar 16000 -ac 1 /kaggle/temp/giong16k.wav
time ../lsenv/bin/python -m scripts.inference \
  --unet_config_path configs/unet/stage2.yaml \
  --inference_ckpt_path checkpoints/latentsync_unet.pt \
  --inference_steps 20 --guidance_scale 1.5 --enable_deepcache \
  --video_path "$(find /kaggle/input -name mat.mp4 | head -1)" \
  --audio_path /kaggle/temp/giong16k.wav \
  --video_out_path /kaggle/temp/noi_raw.mp4
# ghép lại giọng chất lượng cao 44.1kHz thay cho bản 16kHz
ffmpeg -v error -y -i /kaggle/temp/noi_raw.mp4 -i /kaggle/working/giong.wav -map 0:v -map 1:a \
  -c:v libx264 -crf 18 -pix_fmt yuv420p -c:a aac -b:a 192k -shortest /kaggle/working/noi.mp4
ls -lh /kaggle/working
echo "XONG — tải noi.mp4 ở mục Output bên phải"

In [ ]:
from IPython.display import Video
Video(f"{W}/noi.mp4", embed=True, width=360)